In [1]:
!conda install -c conda-forge sentencepiece -y

Retrieving notices: done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.11.0
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/pytorch_p310

  added / updated specs:
    - sentencepiece


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libsentencepiece-0.2.1     |       h1d72405_2         838 KB  conda-forge
    sentencepiece-0.2.1        |       hc5512b6_2          19 KB  conda-forge
    sentencepiece-python-0.2.1 |  py310h1469a80_2         3.4 MB  conda-forge
    sentencepiece-spm-0.2.1    |       h1d72405_2          84 KB  conda-forge
    ------------------------------------------------------------
                                           Total:        

In [2]:
!pip install datasets evaluate "transformers[sentencepiece]" "pyarrow<16"

INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 86.1 MB/s  0:00:006m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 125.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 34.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 138.1 MB/s  0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 170.7 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 20.0.0

In [3]:
!pip install --upgrade "mlflow>=3.1"
!pip install pandas
!pip install scikit-learn
!pip install boto3
!pip install spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 52.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 186.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 84.7 MB/s  0:00:00
  Attempting uninstall: mlflow-tracing
    Found existing installation: mlflow-tracing 3.9.0
    Uninstalling mlflow-tracing-3.9.0:━━━━━━━━━━ 0/3 [mlflow-tracing]
      Successfully uninstalled mlflow-tracing-3.9.0m 0/3 [mlflow-tracing]
  Attempting uninstall: mlflow-skinny━━━━━━━━━━━ 0/3 [mlflow-tracing]
    Found existing installation: mlflow-skinny 3.9.02m0/3 [mlflow-tracing]
    Uninstalling mlflow-skinny-3.9.0:━━━━━━━ 0/3 [mlflow-tracing]
      Successfully uninstalled mlflow-skinny-3.9.0━━━━━━━━━━━━━━━━ 1/3 [mlflow-skinny]
  Attempting uninstall: mlflow╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [mlflow-skinny]
    Found existing installation: mlflow 3.9.0━━━━━━━━━━━━━━━━━ 1/3 [mlflow-skinny]
    Uninstalling mlflow-3.9.0:━━━━╸━━━━━━━━━━━━━ 2/3 [mlflow]ny]
      Successf

In [4]:
!python -m spacy download en_core_web_md

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 123.0 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [5]:
import pandas as pd
import s3fs
import mlflow
from mlflow.tracking import MlflowClient
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score
import spacy
import re
import time

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [6]:
!pip install --upgrade huggingface_hub

from huggingface_hub import login
login()

In [7]:
# =======================================================
# CONFIGURACIÓN DE LA FASE D (Test Set)
# =======================================================
AUTOR_EXPERIMENTO = "Daniel Varela"

# --- Rutas del TEST SET CRUDO ---
S3_BUCKET = "parcial-pln"
TEST_PATH = "parcial-pln/data/raw/test_parquet/" 

TEXT_COLUMN_RAW = "text"
TARGET_COLUMN = "label"

# --- Mapeo de Etiquetas de Hugging Face ---
LABEL_MAPPING = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

# --- MLflow ---
MLFLOW_TRACKING_URI = "http://ec2-52-21-111-46.compute-1.amazonaws.com:5000"
EXPERIMENT_NAME = "Sentimientos403_Comparacion"
REGISTERED_MODEL_NAME = "Sentimientos800" 

# =======================================================
# CONFIGURACIÓN DE SPACY
# =======================================================
USE_LEMMATIZATION = False 
DROP_STOPWORDS = False
DROP_PUNCTUATION = True
NORMALIZE_ELONGATION = True

print("Cargando modelo de spaCy...")
nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])

def limpiar_texto_crudo(text: str) -> str:
    if pd.isna(text): return ""
    text = str(text).lower()
    
    if NORMALIZE_ELONGATION:
        text = re.sub(r'(.)\1{2,}', r'\1', text)
        
    doc = nlp(text)
    tokens = []
    
    for token in doc:
        if DROP_STOPWORDS and token.is_stop: continue
        if DROP_PUNCTUATION and token.is_punct: continue
        word = token.lemma_ if USE_LEMMATIZATION else token.text
        tokens.append(word)
        
    return " ".join(tokens)

Cargando modelo de spaCy...


In [8]:
fs = s3fs.S3FileSystem()

def load_parquet_from_s3(prefix: str) -> pd.DataFrame:
    files = fs.ls(prefix)
    df_list = [pd.read_parquet(f"s3://{file}", filesystem=fs) for file in files]
    return pd.concat(df_list, ignore_index=True)

print("Cargando el Test Set Intocable...")
test_df = load_parquet_from_s3(TEST_PATH)

# Texto crudo directo para Hugging Face
X_test_hf = test_df[TEXT_COLUMN_RAW].fillna("").tolist()
y_test = test_df[TARGET_COLUMN]

print(f"Limpiando {len(X_test_hf)} textos en tiempo real para el Modelo Clásico...")
# Limpieza al vuelo para la Regresión Logística
start_clean = time.time()
X_test_ml = [limpiar_texto_crudo(texto) for texto in X_test_hf]
print(f"Limpieza completada en {time.time() - start_clean:.2f} segundos.")

Cargando el Test Set Intocable...
Limpiando 240000 textos en tiempo real para el Modelo Clásico...
Limpieza completada en 976.50 segundos.


In [9]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

# 1. Tu Campeón Clásico
print("Descargando tu Campeón (Regresión Logística) desde MLflow...")
model_uri = f"models:/{REGISTERED_MODEL_NAME}@champion"
ml_champion = mlflow.sklearn.load_model(model_uri)
print("¡Campeón clásico cargado!")

# 2. El Transformer
print("Descargando modelo DistilBERT de HuggingFace...")
hf_pipeline = pipeline(
    "sentiment-analysis", 
    truncation=True,
    model="distilbert-base-uncased-finetuned-sst-2-english",
    max_length=512,
    device=-1 
)
print("¡Modelo Transformer cargado!")

2026/03/08 02:06:18 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range
2026/03/08 02:06:18 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range
2026/03/08 02:06:18 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


Descargando tu Campeón (Regresión Logística) desde MLflow...


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


¡Campeón clásico cargado!
Descargando modelo DistilBERT de HuggingFace...


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

¡Modelo Transformer cargado!


In [10]:
print("Prediciendo con tu Campeón Clásico...")
start_time = time.time()

y_pred_ml = ml_champion.predict(X_test_ml)

ml_time = time.time() - start_time
acc_ml = accuracy_score(y_test, y_pred_ml)
f1_ml = f1_score(y_test, y_pred_ml, average="macro")

print(f"Tiempo de inferencia: {ml_time:.2f} segundos")
print(f"F1-Score: {f1_ml:.4f}")

Prediciendo con tu Campeón Clásico...
Tiempo de inferencia: 4.64 segundos
F1-Score: 0.8072


In [ ]:
from tqdm.auto import tqdm

print(f"Prediciendo {len(X_test_ml)} textos con Hugging Face...")
start_time = time.time()

hf_results = []
batch_size = 32 # Reducimos un poco el batch para que la CPU no se ahogue

# Creamos un bucle manual con una barra de progreso (tqdm)
for i in tqdm(range(0, len(X_test_ml), batch_size), desc="Procesando lotes"):
    # Tomamos un pedacito de los datos
    batch = X_test_ml[i : i + batch_size]
    
    # Predecimos ese pedacito
    batch_res = hf_pipeline(batch)
    hf_results.extend(batch_res)

hf_time = time.time() - start_time

# Traducimos las etiquetas
y_pred_hf = [LABEL_MAPPING[res["label"]] for res in hf_results]

acc_hf = accuracy_score(y_test, y_pred_hf)
f1_hf = f1_score(y_test, y_pred_hf, average="macro")

print(f"⏱️ Tiempo de inferencia: {hf_time:.2f} segundos")
print(f"📊 F1-Score: {f1_hf:.4f}")

Prediciendo 240000 textos con Hugging Face...


Procesando lotes:   0%|          | 0/7500 [00:00<?, ?it/s]

In [ ]:
print("="*50)
print(" RESULTADOS DE LA COMPARACION (TEST SET)")
print("="*50)
print(f"Regresión Logística (Tu Campeón)  -> F1: {f1_ml:.4f} | Tiempo: {ml_time:.2f}s")
print(f"Hugging Face (DistilBERT)         -> F1: {f1_hf:.4f} | Tiempo: {hf_time:.2f}s")
print("="*50)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    mlflow.create_experiment(name=EXPERIMENT_NAME)
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="Comparativa_TestSet_Final"):
    mlflow.set_tag("Autor", AUTOR_EXPERIMENTO)
    mlflow.set_tag("dataset", "Test_Set_Crudo")
    
    mlflow.log_metric("ML_Classic_F1", f1_ml)
    mlflow.log_metric("ML_Classic_Time_Seconds", ml_time)
    
    mlflow.log_metric("HuggingFace_F1", f1_hf)
    mlflow.log_metric("HuggingFace_Time_Seconds", hf_time)
    
    mlflow.log_metric("Diferencia_F1_HF_vs_ML", f1_hf - f1_ml)

print("Resultados guardados en MLflow.")